# Importance Classifier

Binary sentence-level classifier: **important** (action / decision) vs **not important** (abstract / problem).

**Pipeline:**
1. Load `data/processed/importance_data.json` (produced by `01_data_preparation.ipynb`)
2. Feature engineering — TF-IDF vectors fused with handcrafted linguistic features
3. Train & evaluate four classifiers: Logistic Regression, Random Forest, Decision Tree, KNN
4. Compare with metrics table, confusion matrices, ROC curves, and bar charts
5. Save the best model to `models/importance_classifier.pkl`

## 1. Setup

In [ ]:
import json
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

ROOT      = Path('../')
DATA_PATH = ROOT / 'data' / 'processed' / 'importance_data.json'
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

print('All imports OK')
print(f'Data path : {DATA_PATH}')
print(f'Model dir : {MODEL_DIR}')

## 2. Load & Explore Data

In [ ]:
with open(DATA_PATH, encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
print(f'Total examples : {len(df)}')
print(f'Columns        : {list(df.columns)}')
print(f'Class balance  :')
print(df['label'].value_counts().rename({1: 'Important (1)', 0: 'Not Important (0)'}))
df.head(6)

In [ ]:
# ── Class distribution bar chart ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts().sort_index()
bars = axes[0].bar(['Not Important (0)', 'Important (1)'], counts.values,
                   color=PALETTE[:2], edgecolor='white', linewidth=1.5, width=0.5)
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                 str(v), ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, counts.max() * 1.15)

# Sentence length distribution
df['word_count'] = df['text'].str.split().str.len()
for label, color, name in [(0, PALETTE[0], 'Not Important'), (1, PALETTE[1], 'Important')]:
    axes[1].hist(df[df['label'] == label]['word_count'], bins=30,
                 alpha=0.65, color=color, label=name, edgecolor='white')
axes[1].set_title('Sentence Length Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig(MODEL_DIR / 'importance_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering

Two feature groups are built and fused:
- **TF-IDF** (unigrams + bigrams, top 8 000 features) — captures word importance across corpus
- **Handcrafted linguistic features** — domain-specific signals for meeting importance

In [ ]:
# ── Regex patterns for meeting-domain features ─────────────────────────────────
MODAL_VERBS   = re.compile(r'\b(will|would|shall|should|must|need to|ought to|going to|have to|need|wants to)\b', re.I)
DECISION_KWS  = re.compile(r'\b(decided|agreed|confirmed|resolved|concluded|chosen|approved|rejected|accepted|finalised|finalized|going with|stick with|settle on)\b', re.I)
ACTION_VERBS  = re.compile(r'\b(make|create|build|prepare|write|send|review|check|update|implement|design|discuss|look into|find out|contact|set up|organise|organize|schedule|arrange|follow up)\b', re.I)
NEGATION_KWS  = re.compile(r"\b(not|no|never|without|don't|doesn't|didn't|won't|wouldn't|can't|cannot)\b", re.I)
FIRST_PERSON  = re.compile(r'\b(I|we|our|my|us)\b')
QUESTION_RE   = re.compile(r'\?')


def extract_handcrafted(texts: list) -> np.ndarray:
    """Return (n_samples, n_features) float array of linguistic features."""
    features = []
    for text in texts:
        words = text.split()
        n     = max(len(words), 1)
        row = [
            len(MODAL_VERBS.findall(text)),           # modal verb count
            len(DECISION_KWS.findall(text)),          # decision keyword count
            len(ACTION_VERBS.findall(text)),          # action verb count
            len(NEGATION_KWS.findall(text)),          # negation count
            len(FIRST_PERSON.findall(text)),          # first-person pronoun count
            int(bool(QUESTION_RE.search(text))),      # is a question? (binary)
            len(words),                               # sentence length (words)
            len(text),                                # sentence length (chars)
            len(MODAL_VERBS.findall(text)) / n,       # modal verb density
            sum(1 for w in words if w[0].isupper() and len(w) > 1) / n,  # capitalized ratio
            int(text.endswith('.')),                  # ends with period
            text.count(',') / n,                      # comma density
        ]
        features.append(row)
    return np.array(features, dtype=float)


FEATURE_NAMES_HANDCRAFTED = [
    'modal_count', 'decision_kw_count', 'action_verb_count', 'negation_count',
    'first_person_count', 'is_question', 'word_count', 'char_count',
    'modal_density', 'capital_ratio', 'ends_with_period', 'comma_density',
]

print(f'Handcrafted features ({len(FEATURE_NAMES_HANDCRAFTED)}): {FEATURE_NAMES_HANDCRAFTED}')

In [ ]:
texts  = df['text'].tolist()
labels = df['label'].values

# ── TF-IDF ─────────────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=8_000,
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
)
X_tfidf = tfidf.fit_transform(texts)

# ── Handcrafted ─────────────────────────────────────────────────────────────────
X_hand  = extract_handcrafted(texts)
X_hand_scaled = StandardScaler().fit_transform(X_hand)

# ── Fused feature matrix ────────────────────────────────────────────────────────
X_fused = hstack([X_tfidf, csr_matrix(X_hand_scaled)])

print(f'TF-IDF shape    : {X_tfidf.shape}')
print(f'Handcrafted shape: {X_hand.shape}')
print(f'Fused shape     : {X_fused.shape}')

In [ ]:
# ── Handcrafted feature correlation heatmap ────────────────────────────────────
hand_df = pd.DataFrame(X_hand, columns=FEATURE_NAMES_HANDCRAFTED)
hand_df['label'] = labels

fig, ax = plt.subplots(figsize=(10, 7))
corr = hand_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Handcrafted Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'importance_feature_corr.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_fused, labels, test_size=0.20, random_state=SEED, stratify=labels
)

print(f'Train : {X_train.shape[0]} samples  (pos={y_train.sum()}, neg={(y_train==0).sum()})')
print(f'Test  : {X_test.shape[0]}  samples  (pos={y_test.sum()}, neg={(y_test==0).sum()})')

## 5. Model Training

Four classifiers trained on the fused feature matrix.  
Cross-validation (5-fold, stratified) is used for robust F1 estimation.

In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=3,
        class_weight='balanced', random_state=SEED
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=7, metric='cosine', n_jobs=-1
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    cv_f1 = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1', n_jobs=-1)

    return {
        'model'     : model,
        'name'      : name,
        'y_pred'    : y_pred,
        'y_prob'    : y_prob,
        'accuracy'  : accuracy_score(y_te, y_pred),
        'precision' : precision_score(y_te, y_pred, zero_division=0),
        'recall'    : recall_score(y_te, y_pred, zero_division=0),
        'f1'        : f1_score(y_te, y_pred, zero_division=0),
        'auc'       : roc_auc_score(y_te, y_prob) if y_prob is not None else np.nan,
        'cv_f1_mean': cv_f1.mean(),
        'cv_f1_std' : cv_f1.std(),
        'cm'        : confusion_matrix(y_te, y_pred),
    }


results = {}
for name, model in MODELS.items():
    print(f'Training {name}...', end=' ')
    res = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results[name] = res
    print(f'F1={res["f1"]:.4f}  AUC={res["auc"]:.4f}  CV-F1={res["cv_f1_mean"]:.4f}±{res["cv_f1_std"]:.4f}')

## 6. Results & Visualisations

### 6.1 Metrics Comparison Table

In [ ]:
metrics_df = pd.DataFrame([
    {
        'Model'      : r['name'],
        'Accuracy'   : round(r['accuracy'],  4),
        'Precision'  : round(r['precision'], 4),
        'Recall'     : round(r['recall'],    4),
        'F1'         : round(r['f1'],        4),
        'AUC-ROC'    : round(r['auc'],       4),
        'CV F1 (mean)': round(r['cv_f1_mean'], 4),
        'CV F1 (std)': round(r['cv_f1_std'],  4),
    }
    for r in results.values()
]).set_index('Model').sort_values('F1', ascending=False)

display(metrics_df.style
    .background_gradient(cmap='YlGn', subset=['F1', 'AUC-ROC', 'CV F1 (mean)'])
    .background_gradient(cmap='Oranges', subset=['Precision', 'Recall'])
    .format('{:.4f}')
    .set_caption('Importance Classifier — Model Comparison')
)

### 6.2 Confusion Matrices

In [ ]:
model_names = list(results.keys())
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for ax, (name, res), color in zip(axes, results.items(), PALETTE):
    cm = res['cm']
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    annot = np.array([[f'{cm[i,j]}\n({cm_norm[i,j]:.1%})' for j in range(2)] for i in range(2)])

    sns.heatmap(
        cm_norm, annot=annot, fmt='', cmap='Blues',
        xticklabels=['Not Important', 'Important'],
        yticklabels=['Not Important', 'Important'],
        ax=ax, cbar=False, linewidths=1, linecolor='white',
        annot_kws={'size': 11}
    )
    ax.set_title(f'{name}\nF1={res["f1"]:.4f}  AUC={res["auc"]:.4f}',
                 fontsize=12, fontweight='bold', pad=8)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

fig.suptitle('Confusion Matrices — Importance Classifier', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'importance_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random (AUC = 0.50)', alpha=0.6)

for (name, res), color in zip(results.items(), PALETTE):
    if res['y_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
        ax.plot(fpr, tpr, color=color, linewidth=2.2,
                label=f'{name}  (AUC = {res["auc"]:.4f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Importance Classifier', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig(MODEL_DIR / 'importance_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.4 Metric Bar Chart Comparison

In [ ]:
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
plot_df     = metrics_df[metric_cols].reset_index()
plot_melted = plot_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

fig, ax = plt.subplots(figsize=(13, 5))
bars = sns.barplot(
    data=plot_melted, x='Metric', y='Score', hue='Model',
    palette=PALETTE, ax=ax, edgecolor='white', linewidth=0.8
)

# Annotate bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=7.5, padding=2)

ax.set_ylim(0, 1.12)
ax.set_title('Model Comparison by Metric — Importance Classifier',
             fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'importance_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.5 Cross-Validation F1 with Error Bars

In [ ]:
names    = list(results.keys())
cv_means = [results[n]['cv_f1_mean'] for n in names]
cv_stds  = [results[n]['cv_f1_std']  for n in names]
test_f1  = [results[n]['f1']         for n in names]

x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, cv_means, w, yerr=cv_stds, capsize=5,
            color=PALETTE[0], alpha=0.85, label='CV F1 (5-fold ± std)', edgecolor='white')
b2 = ax.bar(x + w/2, test_f1,  w,
            color=PALETTE[1], alpha=0.85, label='Test F1', edgecolor='white')

ax.bar_label(b1, fmt='%.3f', fontsize=9, padding=3)
ax.bar_label(b2, fmt='%.3f', fontsize=9, padding=3)

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('F1 Score')
ax.set_title('CV F1 vs Test F1 — Importance Classifier',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'importance_cv_vs_test_f1.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.6 Feature Importance

#### Random Forest — Top 25 Features

In [ ]:
rf_model   = results['Random Forest']['model']
tfidf_feat = tfidf.get_feature_names_out().tolist()
all_feat   = tfidf_feat + FEATURE_NAMES_HANDCRAFTED

importances   = rf_model.feature_importances_
feat_imp_df   = pd.DataFrame({'feature': all_feat, 'importance': importances})
feat_imp_df   = feat_imp_df.sort_values('importance', ascending=False).head(25)

fig, ax = plt.subplots(figsize=(9, 8))
colors  = [PALETTE[2] if f in FEATURE_NAMES_HANDCRAFTED else PALETTE[0]
           for f in feat_imp_df['feature']]

ax.barh(feat_imp_df['feature'][::-1], feat_imp_df['importance'][::-1],
        color=colors[::-1], edgecolor='white')
ax.set_xlabel('Feature Importance (Mean Decrease Impurity)', fontsize=11)
ax.set_title('Random Forest — Top 25 Feature Importances\n'
             '(blue = TF-IDF token, green = handcrafted feature)',
             fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE[0], label='TF-IDF token'),
    Patch(facecolor=PALETTE[2], label='Handcrafted feature'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'importance_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

#### Logistic Regression — Top Positive & Negative Coefficients

In [ ]:
lr_model = results['Logistic Regression']['model']
coefs    = lr_model.coef_[0]

coef_df = pd.DataFrame({'feature': all_feat, 'coef': coefs})
top_pos = coef_df.nlargest(15, 'coef')
top_neg = coef_df.nsmallest(15, 'coef')
top_lr  = pd.concat([top_pos, top_neg]).sort_values('coef')

fig, ax = plt.subplots(figsize=(9, 9))
colors  = [PALETTE[1] if c > 0 else PALETTE[3] for c in top_lr['coef']]
ax.barh(top_lr['feature'], top_lr['coef'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient Value', fontsize=11)
ax.set_title('Logistic Regression — Top 15 Positive & Negative Coefficients\n'
             '(orange → Important, red → Not Important)',
             fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE[1], label='Predicts Important'),
    Patch(facecolor=PALETTE[3], label='Predicts Not Important'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'importance_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

#### Decision Tree — Top Feature Importances

In [ ]:
dt_model    = results['Decision Tree']['model']
dt_imp_df   = pd.DataFrame({'feature': all_feat, 'importance': dt_model.feature_importances_})
dt_imp_df   = dt_imp_df.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 7))
colors  = [PALETTE[2] if f in FEATURE_NAMES_HANDCRAFTED else '#8DA0CB'
           for f in dt_imp_df['feature']]
ax.barh(dt_imp_df['feature'][::-1], dt_imp_df['importance'][::-1],
        color=colors[::-1], edgecolor='white')
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Decision Tree — Top 20 Feature Importances', fontsize=13, fontweight='bold')

legend_elements = [
    Patch(facecolor='#8DA0CB', label='TF-IDF token'),
    Patch(facecolor=PALETTE[2], label='Handcrafted feature'),
]
ax.legend(handles=legend_elements, fontsize=10)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'importance_dt_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Classification Reports

In [ ]:
for name, res in results.items():
    print(f'{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['Not Important', 'Important']
    ))

## 8. Best Model Selection & Save

In [ ]:
best_name = metrics_df['F1'].idxmax()
best_res  = results[best_name]

print('╔' + '═'*50 + '╗')
print(f'║  Best Model : {best_name:<35}║')
print(f'║  F1         : {best_res["f1"]:.4f}{" "*29}║')
print(f'║  AUC-ROC    : {best_res["auc"]:.4f}{" "*29}║')
print(f'║  Precision  : {best_res["precision"]:.4f}{" "*29}║')
print(f'║  Recall     : {best_res["recall"]:.4f}{" "*29}║')
print(f'║  Accuracy   : {best_res["accuracy"]:.4f}{" "*29}║')
print('╚' + '═'*50 + '╝')

# Save best model + vectorizer
save_obj = {
    'model'               : best_res['model'],
    'tfidf_vectorizer'    : tfidf,
    'feature_names'       : all_feat,
    'handcrafted_fn_names': FEATURE_NAMES_HANDCRAFTED,
    'best_model_name'     : best_name,
    'metrics'             : {
        'f1'       : best_res['f1'],
        'auc'      : best_res['auc'],
        'precision': best_res['precision'],
        'recall'   : best_res['recall'],
        'accuracy' : best_res['accuracy'],
    },
}

save_path = MODEL_DIR / 'importance_classifier.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(save_obj, f)

print(f'\nModel saved → {save_path}')

## 9. Inference Helper

Quick sanity check — feed new sentences through the saved model.

In [ ]:
def predict_importance(sentences: list, model_path=save_path) -> pd.DataFrame:
    """Load saved model and predict importance for a list of sentences."""
    with open(model_path, 'rb') as f:
        obj = pickle.load(f)

    clf   = obj['model']
    tvec  = obj['tfidf_vectorizer']

    X_tf  = tvec.transform(sentences)
    X_hc  = extract_handcrafted(sentences)
    X_hc  = StandardScaler().fit_transform(X_hc)
    X     = hstack([X_tf, csr_matrix(X_hc)])

    preds = clf.predict(X)
    probs = clf.predict_proba(X)[:, 1] if hasattr(clf, 'predict_proba') else [None]*len(sentences)

    return pd.DataFrame({
        'sentence'  : sentences,
        'prediction': ['Important' if p == 1 else 'Not Important' for p in preds],
        'confidence': [f'{p:.2%}' if p is not None else '—' for p in probs],
    })


test_sentences = [
    "John will send the final report by Friday.",
    "The team decided to use the blue remote control design.",
    "The meeting was held in the conference room.",
    "We agreed on a budget of fifty euros for the prototype.",
    "Someone mentioned that it was cold outside.",
    "Marketing should prepare the presentation slides for next week.",
]

display(predict_importance(test_sentences))